**Human Movement Intelligence | Biomechanical Load Estimation | Presented by: Reza Saadatyar (2026)**

**07 | Multi-Seed Robustness, GRU/BiGRU, Peak-Loss Ablation and Participant-Level Evidence**

![Completed Stage 07 multi-seed robustness with source wearable sensors and the protected participant partitions](../../images/07_consolidated_workflow.svg)

**Research question and completed scope:** This study examined whether compact recurrent and convolutional temporal regressors offer stable original-grid finite-element strain predictions across training seeds and new Training-fold participants, and whether a peak-aware objective improves prediction of source FE peak values and positions.

**Source and study governance:** The source-verified Essex Figshare v13 dataset contains 1,729 eligible trials with original 101×42 IMU measurements and 11×2 unitless model-derived FE strain targets per trial. All new fits and error calculations used only 29 Training participants (1,354 trials), with the same five participant-disjoint folds as Stage 05. The four previously inspected Validation participants were excluded from new model ranking; all four Test participants (183 trials) remained reserved.

**Executed comparison:** Temporal CNN, Dilated TCN, GRU and bidirectional GRU (BiGRU), plus a controlled Dilated TCN without peak loss, each completed **three seeds (2026, 2027, 2028) × five folds**: 75 independent Training fits of 35 epochs. The recorded environment used a Tesla T4 CUDA GPU, detected using the shared computer_device() helper. Model parameter counts differ, so this is a fixed-epoch comparison, not a capacity-matched benchmark.

**Primary finding:** The locked Stage 05 **Random Forest** retained the lowest Training Out-of-Fold subject-balanced MAE (**0.0007950**). Among four main temporal architectures, Dilated TCN ranked first (**0.0008769**), closely followed by BiGRU (**0.0008800**), GRU (**0.0008898**) and Temporal CNN (**0.0009152**). These are development-only results, not final Test findings.

**Peak-loss and sensitivity findings:** The no-peak-loss TCN improved average Training OOF waveform MAE (**0.0008624** versus **0.0008769** with peak weight 0.25); peak-magnitude and peak-index benefits were not consistently attributable to the peak-aware objective. Predeclared fold-mean sensor replacement produced the greatest mean error increase for the right-thigh sensor group (RTH), without establishing causal biomechanical importance.

**Execution:** The simplified cells have not been rerun on the full dataset. Numerical findings described here refer to the previously completed run. Each notebook reads the current upstream files; re-execution replaces its own output files. No historical reports, hash comparisons or PASSED gates are required.


**00. Colab Repository Setup**

Use the shared non-destructive Git synchroniser before importing Python research modules. The script is the same approved generic setup used by earlier Essex notebooks.


In [1]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")

if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}

Repository up to date.
/content/human-movement-intelligence


**01. Imports, Shared Device Detection and Research Environment**

The run used the shared computer_device() helper from the completed HARTH project. The recorded environment was **Tesla T4 (CUDA)** with PyTorch **2.11.0+cu130**, and the same reported GPU device was passed to all 75 Training jobs. The reusable job cache was stored locally in Git-ignored data/processed; fitted model weights and private numeric arrays were not committed.


In [ ]:
# =================== 01. Environment and Experiment Imports ==================

# ------------------------- 1. Study Environment Setup ------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("biomechanical_load_estimation")

# ---------------------------- 2. Standard Library ----------------------------
from pathlib import Path

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
plot_style(
    font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(9, 4),
)

# ------------------- 5. Shared and Study Research Utilities ------------------
from core.output_download import download_outputs_from_colab
from essex_timebase_audit import EXPECTED_CHANNELS
from essex_regression_baselines import (
    regression_metrics, participant_metrics, subject_balanced_mae,
)
from essex_stage07_robustness import (
    ARCHITECTURES, NO_PEAK, CONFIGS, SEEDS, FOLDS, sensor_channel_groups,
    run_training_fold, paired_subject_bootstrap,
)
from core.computer_device import computer_device
from essex_paired_dataset import (
    read_trial_splits,
    load_paired_data,
    read_training_folds,
)

# ----------------------- 6. Source and Output Locations ----------------------
study_dir = root / "biomechanical_load_estimation"
raw_dir = root / "data" / "raw" / "essex_femoral_neck_loading"
mat_dir = raw_dir / "matlab_records"
processed_dir = root / "data" / "processed" / "essex_femoral_neck_loading"
output_dir = study_dir / "output"
job_cache_dir = processed_dir / "stage07_training_jobs"
output_dir.mkdir(parents=True, exist_ok=True)
job_cache_dir.mkdir(parents=True, exist_ok=True)

# ------------------ 7. Shared Hardware Inspection and Device -----------------
device = computer_device()
DEVICE = device.type
print("Predeclared training jobs:", len(CONFIGS) * len(SEEDS) * len(FOLDS))
print("Each training run replaces its current per-fold output files.")


**02. Current Participant Splits and Model Inputs**

Read the current data and display the observed results. Re-running this section replaces its own saved outputs.


In [ ]:
# ======================= 02. Current Participant Splits ======================

# --------------------- 1. Read Current Trial Assignments ---------------------
assigned = read_trial_splits(output_dir)

# --------------------- 2. Prepare Participant Split Masks --------------------
train_mask = assigned["split"].eq("TRAIN").to_numpy()
valid_mask = assigned["split"].eq("VALIDATION").to_numpy()
test_mask = assigned["split"].eq("TEST").to_numpy()
display(
    assigned.groupby("split").agg(
        trials=("local_record_key", "size"), participants=("subject_id", "nunique")
    )
)

# ------------------- 3. Read Current Upstream Model Results ------------------
base_cv_path = output_dir / "05_training_cv_model_summary.csv"
temporal_cv_path = output_dir / "06_training_cv_model_summary.csv"
fold_path = output_dir / "05_training_fold_registry.csv"
prior_subject_path = (
    output_dir / "05_training_cv_subject_activity_metrics.csv"
)
prior_cv = pd.read_csv(base_cv_path)
prior_temporal_cv = pd.read_csv(temporal_cv_path)
prior_rf_cv_mae = float(prior_cv.loc[
    prior_cv["model"].eq("Random Forest"), "subject_balanced_mae"
].iloc[0])


**03. Load Current Paired Arrays**

Read the current data and display the observed results. Re-running this section replaces its own saved outputs.


In [ ]:
# ======================= 03. Load Current Paired Arrays ======================

# --------------- 1. Install and Import the Pinned MATLAB Reader --------------
%pip -q install --only-binary=:all: --no-deps "mat-io==1.0.1"
from matio import load_from_mat

# ----------------------- 2. Load Current Paired Arrays -----------------------
paired = load_paired_data(output_dir, processed_dir, raw_dir, load_from_mat, assigned)

# ------------ 3. Select Arrays for the Required Participant Splits -----------
x_train_source = paired["X_full"][train_mask]
y_train_source = paired["y_fe"][train_mask]
idx_train_source = paired["imu_positions"][train_mask]
print("Loaded current arrays:", {name: value.shape for name, value in paired.items()})


**04. Training Folds and Sensor Groups**

Align the current Notebook 05 fold table to the training trial keys. Each participant stays in one fold.


In [ ]:
# ==================== 04. Original Training-Fold Contract ====================

# -------------------- 1. Prepare Training Trial Identities -------------------
train_info = assigned.loc[train_mask].reset_index(drop=True)
reserved_participants = set(assigned.loc[test_mask, "subject_id"])
validation_participants = set(assigned.loc[valid_mask, "subject_id"])
training_participants = set(train_info["subject_id"])

# --------------- 2. Align the Current Participant Fold Registry --------------
frozen = read_training_folds(output_dir, train_info)
fold_labels = frozen["validation_fold"].to_numpy()
groups = train_info["subject_id"].to_numpy()
folds = {}
for fold_number in FOLDS:
    fit_idx = np.flatnonzero(fold_labels != fold_number)
    held_idx = np.flatnonzero(fold_labels == fold_number)
    folds[fold_number] = (fit_idx, held_idx)

# ----------------- 3. Prepare Source Arrays and Sensor Groups ----------------
x_train = x_train_source
y_train = y_train_source
idx_train = idx_train_source
source_schema = pd.read_csv(
    output_dir / "04_channel_schema.csv"
)
sensors = sensor_channel_groups(EXPECTED_CHANNELS)

# ------------------- 4. Save and Display the Training Folds ------------------
frozen.to_csv(
    output_dir / "07_frozen_training_fold_registry.csv", index=False,
)
print("Frozen Training fold registry:", len(frozen), "trials")
print("Original physical sensor groups:", {k: len(v) for k, v in sensors.items()})
display(frozen.groupby("validation_fold").agg(
    trials=("local_record_key", "size"),
    participants=("subject_id", "nunique"),
))


**05. Multi-Seed, Four-Architecture CV and TCN Ablation**

The planned **75 independent jobs** were completed and recorded as TRAINED: five model configurations, three seeds and five participant-disjoint folds. Each fit used 35 epochs, the fixed optimiser and fold-fit-only normalisation. The output comprises **2,625 actual epoch-loss records**, Out-of-Fold predictions retained in local caches and fold-level metrics committed in CSV form. No previously completed job was reused in this recorded run; compatible caches support later recovery in the same preserved environment.


In [ ]:
# ===================== 05. Full Training-Only Experiment =====================

# --------------- 1. Display the Multi-Seed Experiment Settings ---------------
print("RUN MATRIX:", len(CONFIGS), "configurations ×", len(SEEDS),
      "seeds ×", len(FOLDS), "folds =", len(CONFIGS)*len(SEEDS)*len(FOLDS),
      "jobs")
print("Preset:", CONFIGS, " | seeds:", SEEDS)

# -------------- 2. Train the Configurations on Participant Folds -------------
oof_by_config_seed = {}
job_rows, loss_rows, fold_rows, sensitivity_rows = [], [], [], []
for config in CONFIGS:
    for seed in SEEDS:
        predicted_oof = np.full(y_train.shape, np.nan, dtype=np.float64)
        for fold_number in FOLDS:
            fit_rows, held_rows = folds[fold_number]
            check_sensors = config == "Dilated TCN" and seed == SEEDS[0]
            outcome = run_training_fold(
                job_cache_dir,
                config=config,
                seed=seed,
                fold=fold_number,
                x=x_train,
                y=y_train,
                positions=idx_train,
                fit_rows=fit_rows,
                held_rows=held_rows,
                record_keys=train_info["local_record_key"].to_numpy(),
                participant_ids=groups,
                device=DEVICE,
                sensor_groups=sensors,
                sensitivity=check_sensors,
            )
            predicted_oof[held_rows] = outcome["prediction"]
            job_rows.append(
                {
                    "config": config,
                    "seed": seed,
                    "fold": fold_number,
                    "status": outcome["status"],
                    "fit_trials": outcome["fit_trials"],
                    "held_trials": len(held_rows),
                    "parameters": outcome["parameters"],
                    "device": DEVICE,
                }
            )
            for epoch, loss in enumerate(outcome["history"], start=1):
                loss_rows.append(
                    {
                        "config": config,
                        "seed": seed,
                        "fold": fold_number,
                        "epoch": epoch,
                        "training_loss": float(loss),
                    }
                )
            score = subject_balanced_mae(
                y_train[held_rows], outcome["prediction"], groups[held_rows]
            )
            for metric in regression_metrics(y_train[held_rows], outcome["prediction"]):
                fold_rows.append(
                    {
                        "config": config,
                        "seed": seed,
                        "fold": fold_number,
                        "training_subjects": len(set(groups[fit_rows])),
                        "oof_subjects": len(set(groups[held_rows])),
                        "subject_balanced_mae": score,
                        **metric,
                    }
                )
            for condition, prediction in outcome["perturbed"].items():
                for subject in sorted(set(groups[held_rows])):
                    mask = groups[held_rows] == subject
                    for metric in regression_metrics(
                        y_train[held_rows][mask], prediction[mask]
                    ):
                        sensitivity_rows.append(
                            {
                                "config": config,
                                "seed": seed,
                                "subject_id": subject,
                                "condition": condition,
                                **metric,
                            }
                        )
            print(
                config,
                "|",
                seed,
                "| fold",
                fold_number,
                "|",
                outcome["status"],
                "| subject-balanced MAE",
                f"{score:.7f}",
            )
        oof_by_config_seed[config, seed] = predicted_oof

# ---------------- 3. Save Training Histories and Fold Results ----------------
jobs = pd.DataFrame(job_rows)
history = pd.DataFrame(loss_rows)
fold_metric = pd.DataFrame(fold_rows)
sensitivity = pd.DataFrame(sensitivity_rows)
jobs.to_csv(output_dir / "07_job_manifest.csv", index=False)
history.to_csv(output_dir / "07_training_loss_history.csv", index=False)
fold_metric.to_csv(
    output_dir / "07_training_oof_fold_metrics.csv", index=False,
)
print("Full 75-job OOF matrix complete; cached prediction files stay local.")


**06. Participant-Level Metrics and Repeated-Seed Stability**

**Three-seed Training Out-of-Fold subject-balanced MAE (lower is better):**

| Configuration | Mean MAE | Between-seed SD |
|---|---:|---:|
| Dilated TCN (no peak loss; ablation only) | 0.0008624 | 0.00000136 |
| Dilated TCN | 0.0008769 | 0.00000748 |
| BiGRU | 0.0008800 | 0.00000370 |
| GRU | 0.0008898 | 0.00001797 |
| Temporal CNN | 0.0009152 | 0.00001908 |

The no-peak TCN is a **diagnostic ablation**, not a predeclared primary model candidate. Across the four main neural architectures, Dilated TCN had the lowest mean error; BiGRU was close and showed lower observed across-seed variation. Three seeds measure limited initialisation/shuffle sensitivity; the figures do not demonstrate stable performance across datasets, hardware environments or every possible random seed.


In [ ]:
# ===================== 06. Participant and Seed Stability ====================

# ------------ 1. Calculate Participant, Activity and Seed Metrics ------------
subject_tables, activity_tables, seed_rows = [], [], []
for config in CONFIGS:
    for seed in SEEDS:
        pred = oof_by_config_seed[(config, seed)]
        for subject in sorted(training_participants):
            mask = groups == subject
            for metric in regression_metrics(y_train[mask], pred[mask]):
                subject_tables.append({
                    "config": config, "seed": seed,
                    "subject_id": subject, **metric,
                })
        details = participant_metrics(
            y_train, pred, train_info["subject_id"], train_info["task"],
            model=config, phase="TRAIN_OOF",
        )
        details["seed"] = seed
        activity_tables.append(details)
        targets = regression_metrics(y_train, pred)
        seed_rows.append({
            "config": config, "seed": seed,
            "subject_balanced_mae": subject_balanced_mae(
                y_train, pred, groups,
            ),
            "e1_pointwise_mae": targets[0]["mae"],
            "e3_pointwise_mae": targets[1]["mae"],
            "e1_peak_value_mae": targets[0]["peak_value_mae"],
            "e3_peak_value_mae": targets[1]["peak_value_mae"],
            "e1_peak_index_mae": targets[0]["peak_index_mae"],
            "e3_peak_index_mae": targets[1]["peak_index_mae"],
        })

# -------------------- 2. Summarise Repeated-Seed Stability -------------------
subject_metrics = pd.DataFrame(subject_tables)
activity_metrics = pd.concat(activity_tables, ignore_index=True)
seed_metrics = pd.DataFrame(seed_rows)
summary = seed_metrics.groupby("config", as_index=False).agg(
    mean_oof_subject_mae=("subject_balanced_mae", "mean"),
    sd_across_seeds=("subject_balanced_mae", "std"),
    minimum_seed_mae=("subject_balanced_mae", "min"),
    maximum_seed_mae=("subject_balanced_mae", "max"),
    seeds=("seed", "nunique"),
)

# ------------------- 3. Save and Display Robustness Results ------------------
subject_metrics.to_csv(
    output_dir / "07_training_oof_subject_metrics.csv", index=False,
)
activity_metrics.to_csv(
    output_dir / "07_training_oof_subject_activity_metrics.csv",
    index=False,
)
seed_metrics.to_csv(
    output_dir / "07_training_seed_scores.csv", index=False,
)
summary.to_csv(
    output_dir / "07_training_model_stability_summary.csv", index=False,
)
display(summary.sort_values("mean_oof_subject_mae"))


**07. Training-Only Candidate Decision and Paired Participant Bootstrap**

The predeclared primary decision compared four temporal architectures, averaged over three seeds, with the locked Stage 05 **Random Forest** Training-CV result. Random Forest remained first (**0.0007950**), followed by Dilated TCN (**0.0008769**), BiGRU (**0.0008800**), GRU (**0.0008898**) and Temporal CNN (**0.0009152**). The no-peak-loss TCN was excluded from this primary selection pool as specified before execution.

Exploratory paired participant-cluster bootstrap differences (temporal model minus Random Forest; lower is better) were positive for each temporal model. For Dilated TCN, the mean paired difference was **0.00008184**, with an exploratory 95% bootstrap interval **[0.00003462, 0.00012996]**, based on 29 Training participants and 2,000 resamples. These intervals describe development-cohort variation and are **not confirmatory independent-cohort significance tests**.

**Provisional Stage 08 candidate: Random Forest.** No Validation or Test observation entered this decision. Notebook 08 retains the selected Random Forest primary and the predefined descriptive TCN secondary.


In [ ]:
# ============ 07. Training-Only Candidate and Cluster Uncertainty ============

# ----------- 1. Read the Current Random Forest Participant Results -----------
rf_candidates = pd.read_csv(
    output_dir / "05_training_cv_subject_activity_metrics.csv"
)
rf_candidates = rf_candidates.loc[
    rf_candidates["model"].eq("Random Forest")
].copy()
rf_by_person = {}
for subject, part in rf_candidates.groupby("subject_id"):
    rf_by_person[subject] = float(np.average(
        part["mae"].to_numpy(dtype=float),
        weights=part["trials"].to_numpy(dtype=float),
    ))

# ---------------- 2. Combine Repeated-Seed Participant Scores ----------------
def per_subject_model(name):
    data = subject_metrics.loc[subject_metrics["config"].eq(name)]
    return data.groupby("subject_id")["mae"].mean()

# ------------------ 3. Compare Candidates Using Training CV ------------------
rf_series = pd.Series(rf_by_person, dtype=float).sort_index()
primary_options = summary.loc[
    summary["config"].isin(ARCHITECTURES),
    ["config", "mean_oof_subject_mae"]
].rename(columns={
    "config": "candidate",
    "mean_oof_subject_mae": "selection_mae",
})
primary_options = pd.concat([
    primary_options,
    pd.DataFrame([{
        "candidate": "Random Forest (Stage 05 locked)",
        "selection_mae": prior_rf_cv_mae,
    }]),
], ignore_index=True)
primary_options = primary_options.sort_values(
    ["selection_mae", "candidate"], kind="stable",
).reset_index(drop=True)
candidate_stage08 = str(primary_options.iloc[0]["candidate"])
primary_options["training_cv_selected"] = (
    primary_options["candidate"].eq(candidate_stage08)
)

# ------------ 4. Calculate Paired Participant Bootstrap Intervals ------------
comparisons = []
for config in ARCHITECTURES:
    model = per_subject_model(config)
    model = model.reindex(rf_series.index)
    stats = paired_subject_bootstrap(
        model.to_numpy(), rf_series.to_numpy(), draws=2000, seed=2026
    )
    comparisons.append(
        {
            "comparison": f"{config} minus locked RF",
            "metric": "per-participant waveform MAE averaged across two FE targets",
            **stats,
        }
    )
bootstrap = pd.DataFrame(comparisons)

# ------------------ 5. Save Candidate and Bootstrap Results ------------------
bootstrap.to_csv(
    output_dir / "07_paired_subject_bootstrap.csv", index=False,
)
primary_options.to_csv(
    output_dir / "07_training_candidate_decision.csv", index=False,
)
print("Training-only provisional Stage 08 candidate:", candidate_stage08)
print("No Validation or Test outcome entered this decision.")
display(primary_options)
display(bootstrap)


**08. Peak-Loss Ablation and Sensor/Noise Sensitivity**

**Peak-loss ablation:** Across three matched seeds and the same five folds, Dilated TCN without the peak penalty had lower mean Training OOF waveform MAE (**0.0008624**) than the version with peak weight 0.25 (**0.0008769**). Mean e1/e3 peak-value MAEs were **0.0008419 / 0.0018276** without the peak penalty versus **0.0008448 / 0.0018320** with it. Peak-index MAEs also tended to favour the unpenalised version. Improvements were not consistent across every target and seed; the paired participant bootstrap interval for the waveform difference crossed zero. Thus the specified penalty **was not shown to provide a reliable peak-performance benefit**.

**Controlled sensor sensitivity:** Using Dilated TCN seed 2026 on Training OOF predictions, replacing the original right-thigh (RTH) sensor's six channels with their fold-Training mean increased the average e1/e3 MAE by **0.0001060**, the largest observed increase among seven groups. The fixed 0.10 standardised Gaussian perturbation changed the average MAE by approximately **0.00000171**. These are strictly model-specific, artificial-input responses, not causal sensor contributions, physical failure simulations or recommendations for sensor removal.


In [ ]:
# ================ 08. Peak-Loss Ablation and Signal Robustness ===============

# -------------------- 1. Matched Three-Seed Peak Ablation --------------------
tc0 = seed_metrics.loc[
    seed_metrics["config"].eq(NO_PEAK)
].set_index("seed")
tc1 = seed_metrics.loc[
    seed_metrics["config"].eq("Dilated TCN")
].set_index("seed")
ablation_rows = []
for metric in (
    "subject_balanced_mae", "e1_peak_value_mae", "e3_peak_value_mae",
    "e1_peak_index_mae", "e3_peak_index_mae",
):
    for seed in SEEDS:
        ablation_rows.append({
            "metric": metric, "seed": seed,
            "no_peak_weight_0": float(tc0.loc[seed, metric]),
            "peak_weight_0p25": float(tc1.loc[seed, metric]),
            "difference_with_peak_minus_no_peak": float(
                tc1.loc[seed, metric] - tc0.loc[seed, metric]
            ),
        })
ablation = pd.DataFrame(ablation_rows)
ablation.to_csv(
    output_dir / "07_peak_loss_ablation.csv", index=False,
)
tc0_subject = per_subject_model(NO_PEAK).reindex(rf_series.index)
tc1_subject = per_subject_model("Dilated TCN").reindex(rf_series.index)
paired_ablation = paired_subject_bootstrap(
    tc1_subject.to_numpy(), tc0_subject.to_numpy(),
    draws=2000, seed=2027,
)
bootstrap = pd.concat([
    bootstrap,
    pd.DataFrame([{
        "comparison": "Dilated TCN with peak loss minus no-peak TCN",
        "metric": "per-participant waveform MAE averaged over 3 seeds",
        **paired_ablation,
    }]),
], ignore_index=True)
bootstrap.to_csv(
    output_dir / "07_paired_subject_bootstrap.csv", index=False,
)
print("Paired TCN waveform MAE (with peak loss minus without):",
      paired_ablation)
display(ablation.groupby("metric", as_index=False).agg(
    no_peak_mean=("no_peak_weight_0", "mean"),
    with_peak_mean=("peak_weight_0p25", "mean"),
    mean_delta=("difference_with_peak_minus_no_peak", "mean"),
))

# ----------------- 2. First-Seed OOF Sensor/Noise Robustness -----------------
source_clean = subject_metrics.loc[
    subject_metrics["config"].eq("Dilated TCN")
    & subject_metrics["seed"].eq(SEEDS[0]),
    ["subject_id", "target", "mae"],
].rename(columns={"mae": "clean_oof_mae"})
sensitivity = sensitivity.merge(
    source_clean, on=["subject_id", "target"],
    validate="many_to_one",
)
sensitivity["delta_mae"] = (
    sensitivity["mae"] - sensitivity["clean_oof_mae"]
)
sensitivity.to_csv(
    output_dir / "07_sensor_noise_sensitivity.csv", index=False,
)
sensitivity_summary = sensitivity.groupby(
    "condition", as_index=False
).agg(
    mean_oof_mae=("mae", "mean"),
    clean_reference_mae=("clean_oof_mae", "mean"),
    delta_mae=("delta_mae", "mean"),
    participants=("subject_id", "nunique"),
)
print("These perturbations do not estimate causal sensor importance.")
display(sensitivity_summary.sort_values("delta_mae", ascending=False))


**09. Robustness Figures and Activity/Participant Heterogeneity**

All four saved scientific figures are based on genuine Training Out-of-Fold evidence: (1) three-seed model MAE with across-seed variability and the locked Stage 05 RF reference, (2) grouped-bar peak-loss ablation for e1/e3, (3) seven-source-sensor mean replacement and fixed noise sensitivity, and (4) between-participant MAE distributions. Differences among participants and activities must be interpreted as descriptive development diagnostics; these figures do not contain fresh Validation or Test evaluations.


In [ ]:
# ======================== 09. Research Result Figures ========================

# ------------------------ 1. Three-Seed OOF Comparison -----------------------
base = summary.loc[summary["config"].isin(ARCHITECTURES)]

base = base.set_index("config").reindex(ARCHITECTURES)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(
    np.arange(len(base)),
    base["mean_oof_subject_mae"].to_numpy(),
    yerr=base["sd_across_seeds"].to_numpy(),
    capsize=4,
)
ax.axhline(
    prior_rf_cv_mae, linestyle="--", color="black",
    label="Locked Stage 05 Random Forest CV",
)
ax.set_xticks(np.arange(len(base)), base.index, rotation=15)
ax.set_ylabel("Subject-balanced OOF MAE (unitless FE)")
ax.set_title("Three-seed robustness: participant-disjoint Training CV")
ax.legend()
save_figure(fig, output_dir / "07_oof_robustness_comparison.png")
plt.show()

# --------------------- 2. Matched Peak Magnitude Ablation --------------------
peak_view = ablation.loc[
    ablation["metric"].isin(["e1_peak_value_mae", "e3_peak_value_mae"])
].groupby("metric", as_index=True)[
    ["no_peak_weight_0", "peak_weight_0p25"]
].mean().rename(columns={
    "no_peak_weight_0": "TCN without peak loss",
    "peak_weight_0p25": "TCN with peak loss",
})
fig, ax = plt.subplots(figsize=(9, 4))
peak_view.plot.bar(ax=ax, width=0.75)
ax.set_title("Training OOF peak-value error: TCN loss ablation")
ax.set_xlabel("Original FE target")
ax.set_ylabel("Peak-value MAE (unitless FE)")
ax.tick_params(axis="x", rotation=0)
save_figure(fig, output_dir / "07_peak_loss_ablation.png")
plt.show()

# ------------------ 3. Source Sensor and Noise Perturbations -----------------
sens_plot = sensitivity_summary.sort_values("delta_mae", ascending=True)
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(sens_plot["condition"], sens_plot["delta_mae"])
ax.axvline(0.0, color="black", linewidth=1)
ax.set_xlabel("Δ subject-averaged OOF MAE versus unperturbed input")
ax.set_title("Fold-mean sensor replacement and standardised noise")
save_figure(fig, output_dir / "07_sensor_noise_sensitivity.png")
plt.show()

# ------------------- 4. Participant Heterogeneity by Model -------------------
subject_fig = subject_metrics.loc[
    subject_metrics["config"].isin(ARCHITECTURES)
].groupby(["config", "subject_id"])["mae"].mean().reset_index()
positions = range(len(ARCHITECTURES))
fig, ax = plt.subplots(figsize=(10, 4))
values = [
    subject_fig.loc[subject_fig["config"].eq(name), "mae"].to_numpy()
    for name in ARCHITECTURES
]
ax.boxplot(values, tick_labels=ARCHITECTURES, showfliers=True)
ax.set_ylabel("Mean subject FE waveform MAE")
ax.set_title("Between-participant error heterogeneity (Training OOF)")
save_figure(fig, output_dir / "07_subject_error_heterogeneity.png")
plt.show()


**10. Results Summary and Output Download**

Read the current data and display the observed results. Re-running this section replaces its own saved outputs.


In [ ]:
# ================== 10. Results Summary and Output Download ==================

# ------------------------- 1. Display Current Results ------------------------
display(summary)
print("Completed training jobs:", len(jobs))
print("Current robustness results saved in output/ with prefix 07_.")

# -------------------- 2. Download Current Notebook Outputs -------------------
download_outputs_from_colab(
    folder="biomechanical_load_estimation/output", stage="07",
)


**11. Scientific Interpretation, Decision Boundary and Stage 08 Preparation**

**Completed results and model choice.** Across three predeclared seeds, 35-epoch fits and five subject-disjoint Training folds, Random Forest from Stage 05 retained the best overall waveform Training-CV subject-balanced MAE (**0.0007950**). The best principal temporal architecture was **Dilated TCN (0.0008769)**, narrowly ahead of **BiGRU (0.0008800)**, followed by **GRU (0.0008898)** and **Temporal CNN (0.0009152)**. These comparisons use source-aligned, model-derived, unitless original FE targets. They are limited to the Essex development cohort, with no newly used Validation or Test data.

**Stability and architectural interpretation.** Dilated TCN and BiGRU had small observed mean-error differences, and BiGRU showed smaller between-seed variation in the three tested seeds. GRU and Temporal CNN exhibited greater variation. Training budgets were matched in epochs and optimiser settings, but architecture parameter counts and computational complexity were not matched. No ranking here proves a universal superiority of convolutional or recurrent networks.

**What peak-loss ablation actually showed.** Removing the 0.25 peak-loss term lowered Dilated TCN's mean Training OOF waveform MAE from **0.0008769** to **0.0008624** and slightly improved average e1/e3 peak-magnitude and peak-location errors. Per-seed results were mixed, and the exploratory paired participant bootstrap interval for the waveform difference included zero. Therefore the Stage 06 diagnostic Validation peak advantage cannot confidently be attributed to adding this peak-loss term. The ablation was explicitly excluded from the Stage 07 primary architecture selection rule.

**Sensor and participant evidence.** The most disruptive of seven fixed sensor-group mean replacements was right thigh (RTH), with **+0.0001060** mean Training OOF MAE for the specified Dilated TCN seed 2026. Standardised Gaussian noise with SD 0.10 led to a much smaller mean increase (**+0.00000171**) under the tested protocol. Neither result is a causal importance estimate or an independent assessment of real sensor dropout. The paired participant-cluster intervals remain exploratory and reflect only 29 Training participants.

**Next stage:** Notebook 08 fits the existing Random Forest primary and descriptive TCN secondary on Development data, then reports held-out Test results. Test outcomes are not used to change model selection.
